In [2]:
import os
os.chdir('./stat_csv')
os.getcwd()

FileNotFoundError: [Errno 2] No such file or directory: './stat_csv'

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

In [ ]:
from sklearn.model_selection import train_test_split

### Load Dataset

In [ ]:
orig_df = pd.read_excel("Dry_Bean_Dataset.xlsx")
orig_df.head(10)

In [ ]:
orig_df.columns

In [ ]:
orig_df.dtypes

In [ ]:
orig_df['Class'].unique()

In [ ]:
df = orig_df.copy()

In [ ]:
orig_X = df.drop(['Class'], axis=1)
orig_X.columns

In [ ]:
# Finding numerical variables
num_features = orig_X.select_dtypes(include=["int64", "float64"]).columns.tolist()
num_features

### Scaling

In [ ]:
# Scale by standardized normal distribution, (x-mean)/sd
def scale_standard(X_train, X_test):
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled

# Scale by min-max, (x-min)/(max-min)
def scale_minmax(X_train, X_test):
    from sklearn.preprocessing import MinMaxScaler
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled

# Scale by robust, (x-median)/iqr
def scale_minmax(X_train, X_test):
    from sklearn.preprocessing import RobustScaler
    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled

### Encoding Categorical Data

In [ ]:
# One-Hot Encoder version 1: convert all categorical variables
def onehot_encoder_all(df):
    from sklearn.preprocessing import OneHotEncoder
    onehot_encoder = OneHotEncoder(sparse_output=False,        # use sparse=True for large data
                                    handle_unknown='ignore')
    oh_encoded = onehot_encoder.fit_transform(df)
    ohe_df = pd.DataFrame(oh_encoded,
                         columns=onehot_encoder.get_feature_names_out(df.columns))
    return ohe_df

# One-Hot Encoder version 2: Convert selected variables
def onehot_encoder_selected(df, encoded_vars):
    df = pd.get_dummies(df, columns=encoded_vars, drop_first=False)
    return df

# Label Encoder
def label_encoder(df, encoded_var):
    from sklearn.preprocessing import LabelEncoder
    label_encoder = LabelEncoder()
    df[encoded_var] = label_encoder.fit_transform(df[encoded_var])
    return df

# Ordinal Encoder
def ordinal_encoder(df, encoded_var, encode_order):
    from sklearn.preprocessing import OrdinalEncoder
    ordinal_encoder = OrdinalEncoder(categories=[encode_order])
    df[encoded_var] = ordinal_encoder.fit_transform(df[[encoded_var]])
    return df

In [ ]:
# Frequenct Encoder
def frequent_encoder(df, encoded_var):
    freq_encoding = df[encoded_var].value_counts() 
    df[encoded_var] = df[encoded_var].map(freq_encoding)
    return df

# Target Encoder
def target_encoder(df, encoded_var, encoded_num):
    import category_encoders as ce
    target_encoder = ce.TargetEncoder(cols=[encoded_var])
    df[encoded_var] = target_encoder.fit_transform(df[encoded_var], df[encoded_num])
    return df

# Binary Encoder
def binary_encoder(df, encoded_var):
    import category_encoders as ce
    binary_encoder = ce.BinaryEncoder(cols=[encoded_var])
    df_binary = binary_encoder.fit_transform(df[encoded_var])
    df_binary.columns = [f"{encoded_var}_bin_{i}" for i in range(df_binary.shape[1])]
    df = pd.concat([df, df_binary], axis=1)
    df = df.drop([encoded_var], axis=1)
    return df 

### Hash Encoder
def hash_encoder(df, encoded_var, hash_len):
    import category_encoders as ce
    hash_encoder = ce.HashingEncoder(cols=[encoded_var], n_components=hash_len)  # n_components = length of hash, default = 8
    df_hash = hash_encoder.fit_transform(df[encoded_var])
    df_hash.columns = [f"{encoded_var}_hash_{i}" for i in range(df_hash.shape[1])]
    df = pd.concat([df, df_hash], axis=1)
    df = df.drop([encoded_var], axis=1)
    return df 

### Handling Imbalance

In [ ]:
# Undersampling
def undersampling(X,y):
    from imblearn.under_sampling import RandomUnderSampler
    X_res, y_res = RandomUnderSampler().fit_resample(X, y)
    return X_res, y_res

# Oversampling
def oversampling(X,y):
    from imblearn.over_sampling import RandomOverSampler
    X_res, y_res = RandomOverSampler().fit_resample(X, y)
    return X_res, y_res

# SMOTE
def smote(X,y):
    from imblearn.over_sampling import SMOTE
    X_res, y_res = SMOTE().fit_resample(X, y)
    return X_res, y_res

### Performance Evaluation

In [ ]:
# Function to compute performance classification metrics
def evaluate_performance(y_true, y_pred, average_val='macro'):

    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average=average_val)
    recall = recall_score(y_true, y_pred, average=average_val)
    f1 = f1_score(y_true, y_pred, average=average_val)

    from sklearn.metrics import confusion_matrix
    confusion = confusion_matrix(y_true, y_pred)

    return accuracy, precision, recall, f1, confusion

In [ ]:
def plot_roc_auc(y_true, y_pred):
    from sklearn.metrics import roc_curve, roc_auc_score
    import matplotlib.pyplot as plt

    fpr, tpr, thresholds = roc_curve(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred)

    plt.plot(fpr, tpr)
    plt.plot([0, 1], [0, 1], linestyle='--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve (AUC = {auc:.3f})")
    plt.show()

    return auc

### Check Feature Importance

In [ ]:
def get_permutation_importance(model, X, X_true, y_true):
    from sklearn.inspection import permutation_importance
    result = permutation_importance(model, X_test, y_test, n_repeats=10)
    
    perm_df = pd.DataFrame({
        "feature": X.columns,
        "importance_mean": result.importances_mean,
        "importance_std": result.importances_std
    }).sort_values("importance_mean", ascending=False)
    
    return perm_df

    # Higher value = more important feature, relatively compared to other features
    # Near-zero value = unimportant feature, relatively compared to other features
    # Negative value = noisy feature

In [ ]:
def create_shap_values_plot(model, X, X_train, X_test):
    import shap
    X_train_df = pd.DataFrame(X_train, columns=X.columns.tolist())
    X_test_df = pd.DataFrame(X_test, columns=X.columns.tolist())
    if len(X_train) > 200:
        explainer = shap.KernelExplainer(model.predict, X_train_df.sample(200))
    else: 
        explainer = shap.KernelExplainer(model.predict, X_train_df)
    if len(X_test) > 100:
        shap_values = explainer.shap_values(X_test_df.sample(100))
        shap.summary_plot(shap_values, X_test_df.sample(100))
    else:
        shap_values = explainer.shap_values(X_test_df)
        shap.summary_plot(shap_values, X_test_df)

### Experiments: Interpretability

In [ ]:
## Ordinal -> Label Encoder
encoded_var = 'Class'
df = label_encoder(df, encoded_var)
df.head()

In [ ]:
orig_y = df['Class']
orig_y 

In [ ]:
X = orig_X.copy()
y = orig_y.copy()

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

############## Imbalance Handling here ##############
# Smote Example
X_train, y_train = smote(X_train,y_train)
print(X_train.shape, y_train.shape)
y_train.value_counts()
#######################################################

base_X_train = X_train.copy()
base_X_test = X_test.copy()

base_X_train.head()

### Baseline Model

In [ ]:
X_train = base_X_train.copy()
X_test = base_X_test.copy()

# Scaling is not needed for tree-based classification method

#######################################################
# Construct a model
from sklearn.tree import DecisionTreeClassifier
model = DecisionTreeClassifier(criterion = 'gini', random_state = 0)
model.fit(X_train, y_train)
#######################################################

# Use model for predicted responses
y_pred = model.predict(X_test)

acc, precision, recall, f1, confusion = evaluate_performance(y_test, y_pred)
print(confusion)
print(acc, precision, recall, f1)

In [ ]:
# Get summary of classification report
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))

In [ ]:
# Question: Rank classes that perform the best to the worst based on the results above.
# Answer and provide the reason: 

In [ ]:
#### Find average AUC for each class
#### ROC–AUC for multiclass classification is computed using a one-vs-rest strategy.

In [ ]:
y_test

In [ ]:
# Treat every class as binary => Binarize labels in a one-vs-rest
num_classes = 7
from sklearn.preprocessing import label_binarize
y_bin = label_binarize(y_test, classes=list(range(0, num_classes, 1)))  
print(y_bin)
print(y_bin.shape)

In [ ]:
# Derive predicted probabilities
y_pred = model.predict_proba(X_test)
print(y_pred)
print(y_pred.shape)

In [ ]:
# Compute AUC 
fpr = dict()
tpr = dict()
roc_auc = dict()

for i in range(num_classes):
    from sklearn.metrics import roc_curve, auc
    fpr[i], tpr[i], _ = roc_curve(y_bin[:, i], y_pred[:,i])
    roc_auc[i] = auc(fpr[i], tpr[i])

fpr["micro"], tpr["micro"], _ = roc_curve(y_bin.ravel(), y_pred.ravel())  # treat every class as binary and flatten
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])
# Micro AUC indicates how good is the model overall, weighted by class frequency
# If micro-AUC is high, the model is good at ranking the correct class higher than incorrect ones
#                       performance is strong on frequent classes

# Macro-average: average AUC across classes
roc_auc["macro"] = np.mean(list(roc_auc.values()))  # Averages AUC for each class equally
# Macro AUC indicates how well the model performs on an average class
# If macro-AUC is low, the model struggles on minority classes
roc_auc

In [ ]:
plt.figure(figsize=(8,6))

colors = ['blue', 'red', 'green', 'magenta', 'cyan', 'yellow', 'orange']

for i, color in zip(range(num_classes), colors):
    plt.plot(fpr[i], tpr[i], color=color,
             label=f'Class {i} (AUC = {roc_auc[i]:.2f})')

plt.plot([0,1], [0,1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multi-class ROC')
plt.legend()
plt.show()

In [ ]:
# Question: Rank the classes from the best to the worst based on AUC for each class.
#           Compare this ranking with the ranking obtained from previous question (based on precision, recall, F1-scores), 
#           are the rankings from (1) AUC and (2) precision/recall/F1-scores generally similar or noticeably different?
# Answer and provide the reason: Class: 1 > 4 > 5 > 2 > 0 > 3 > 6
# 0: 0.9409647329056837
# 1: 1.0
# 2: 0.9480242611137771
# 3: 0.9201208228386831
# 4: 0.9615826028035405
# 5: 0.9524323134492626
# 6: 0.8980935514471537

In [ ]:
get_permutation_importance(model, base_X_train, X_test, y_test)

In [ ]:
create_shap_values_plot(model, base_X_train, base_X_train, X_test)

### Cross Validation

In [ ]:
############## Imbalance Handling here ##############
# Smote Example
#X_train, y_train = smote(X_train,y_train)
#print(X_train.shape, y_train.shape)
#y_train.value_counts()
#######################################################

In [ ]:
#### KFold + Cross Validation
X = orig_X.copy()
y = orig_y.copy()

# Scaling is not needed for tree-based classification method

#######################################################
# Construct a model
from sklearn.tree import DecisionTreeClassifier
model = DecisionTreeClassifier(criterion = 'gini', class_weight='balanced', random_state = 0)  
# Note: class_weight='balanced' = handling imbalance classes here
#######################################################

#######################################################
# Set up KFold + Cross Validation 
from sklearn.model_selection import KFold, cross_validate
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
#######################################################

scoring = {
    'accuracy': 'accuracy',
    'precision_weighted': 'precision_weighted',
    'recall_weighted': 'recall_weighted',
    'f1_weighted': 'f1_weighted'
}

#######################################################
# Train and run models using KFold + Cross Validation option
cv_results = cross_validate(model, X, y, cv=kfold, scoring=scoring)
#######################################################

for metric in scoring.keys():
    print(f"{metric}: {cv_results['test_'+metric].mean():.4f}")

cv_results

In [ ]:
#### StratifiedKFold + Cross Validation

X = orig_X.copy()
y = orig_y.copy()

# Scaling is not needed for tree-based classification method

#######################################################
# Construct a model
from sklearn.tree import DecisionTreeClassifier
model = DecisionTreeClassifier(criterion = 'gini', class_weight='balanced', random_state = 0)
# Note: class_weight='balanced' = handling imbalance classes here
#######################################################

#######################################################
# Set up StratifiedKFold + Cross Validation
from sklearn.model_selection import StratifiedKFold, cross_validate
stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
#######################################################

scoring = {
    'accuracy': 'accuracy',
    'precision_weighted': 'precision_weighted',
    'recall_weighted': 'recall_weighted',
    'f1_weighted': 'f1_weighted'
}

#######################################################
# Train and run models using StratifiedKFold + Cross Validation option
cv_results = cross_validate(model, X, y, cv=stratified_kfold, scoring=scoring)
#########################################################

#######################################################
#  Get performance of all folds at once
for metric in scoring.keys():
    print(f"{metric}: {cv_results['test_'+metric].mean():.4f}")

cv_results

In [ ]:
# Question: Compare accuracy, precision, recall, F1-scores of models using
#           (1) KFold or (2) StratifiedKFold 
#           which one (1) or (2) has better performance?
# Answer and provide the reason:

In [ ]:
#### StratifiedKFold + Cross Validation
#### Get performance of each fold separately 

X_train = base_X_train.copy()
X_test = base_X_test.copy()

# Scaling is not needed for tree-based classification method

#######################################################
# Construct a model
from sklearn.tree import DecisionTreeClassifier
model = DecisionTreeClassifier(criterion = 'gini', class_weight='balanced', random_state = 0)
# Note: class_weight='balanced' = handling imbalance classes here
#######################################################

#######################################################
# Set up StratifiedKFold + Cross Validation
from sklearn.model_selection import StratifiedKFold, cross_validate
stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
#######################################################

scoring = {
    'accuracy': 'accuracy',
    'f1_weighted': 'f1_weighted',
    'precision_weighted': 'precision_weighted',
    'recall_weighted': 'recall_weighted'
}

#######################################################
# Train and run models for each fold separately
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

for fold, (train_idx, test_idx) in enumerate(stratified_kfold.split(X, y), 1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc, precision, recall, f1, confusion = evaluate_performance(y_test, y_pred)
    print(f"Fold {fold}: ")
    print(confusion)
    print(acc, precision, recall, f1)
#########################################################


### Analaysis of Categorical Data

#### Independence Test

In [ ]:
# Prove: Is bean Class independent of Area size category?
# H0: X and Y are independent 
# H1: X and Y are not independent
# Decision: if p < 0.05 → Reject H0 -> X and Y are not independent -> They are association

In [ ]:
# Categorize Area size from numerical data to categorical data
df['Area_bin'] = pd.qcut(
    df['Area'],
    q=4,
    labels=['Small', 'Medium-Small', 'Medium-Large', 'Large']
)

In [ ]:
contingency_table = pd.crosstab(df['Area_bin'], df['Class'])
print(contingency_table)

In [ ]:
# Chi-square test
from scipy.stats import chi2_contingency
chi2, p, dof, expected = chi2_contingency(contingency_table)

print("Chi-square:", chi2)
print("p-value:", p)

In [ ]:
# Question: Is bean Class independent of Area size category?
# Answer and provide the reason: 

#### Homogeneity Test

In [ ]:
# Prove: Do distribution of AspectRatio_bin is the same across bean Classes?
# H0: P1j = P2j = ... Pkj  (Distributions of category j among groups 1, 2, ..., k are the same.)
# H1: At least one group does not have the same distribution
# Decision: If p < 0.05 → Reject H0 → Distributions differ

In [ ]:
# Categorize Aspect Ratio size from numerical data to categorical data
df['AspectRatio_bin'] = pd.qcut(
    df['AspectRation'],
    q=3,
    labels=['Low', 'Medium', 'High']
)

In [ ]:
# Contingency table (rows = groups, columns = categories)
contingency_table = pd.crosstab(df['Class'], df['AspectRatio_bin'])
print(contingency_table)

In [ ]:
# Chi-square test
from scipy.stats import chi2_contingency
chi2, p, dof, expected = chi2_contingency(contingency_table)

print("Chi-square:", chi2)
print("p-value:", p)

In [ ]:
# Question: Do distribution of AspectRatio_bin is the same across bean Classes?
# Answer and provide the reason:

### Analysis of Variance: Single Factor

In [ ]:
# Prove: Does the mean of Area is the same for all bean Classes?
# H0: m1 = m2 = ... = mJ (means of each cateogory are the same.)
# H1: At least one pair is not the same
# Decision: if p < alpha (significant level) → Reject H0 

In [ ]:
# Perform Anova test
from statsmodels.formula.api import ols
model = ols('Area ~ C(Class)', data=df).fit()

# ANOVA table
import statsmodels.api as sm
anova_table = sm.stats.anova_lm(model, typ=2)
print(anova_table)

In [ ]:
df.groupby('Class')['Area'].mean()

In [ ]:
# Question: Does the mean of Area is the same for all bean Classes?
# Answer and provide the reason:

### Analysis of Variance: Two Factors

In [ ]:
# Prove: Does mean Perimeter (Y) differ by bean Class (A) and Area level (B), and is there an interaction between them?
# IF p < alpha (significant level) → Reject H0 → statistically significant effect
# Reject H0AB: C(A):C(B) → Interaction affects Y
# Reject H0A: C(A) → A affects Y
# Reject H0B: C(B) → B affects Y

In [ ]:
# Factor 1: Class
df['Class'] = df['Class'].astype('category')

# Factor 2: Area (binned)
df['Area_bin'] = pd.qcut(
    df['Area'],
    q=3,
    labels=['Small', 'Medium', 'Large']
)

In [ ]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

model = ols(
    'Perimeter ~ C(Class) + C(Area_bin) + C(Class):C(Area_bin)',
    data=df
).fit()

anova_table = sm.stats.anova_lm(model, typ=2)
print(anova_table)

In [ ]:
df.groupby('Class')['Perimeter'].mean()

In [ ]:
df.groupby('Area_bin')['Perimeter'].mean()

In [ ]:
# Question: Is there an interaction between Class (A) and Area (B)?
# Answer and provide the reason:

In [ ]:
# Question: Does the mean of Perimeter differ by Class (A)?
# Answer and provide the reason:

In [ ]:
# Question: Does the mean of Perimeter differ by Area (B)?
# Answer and provide the reason: